In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


# Interpolate Graph Demo
Load PubChem assay graphs and render a small sample.


In [ ]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import warnings


In [ ]:
from abstractgraph_graphicalizer.chem import PubChemLoader

loader = PubChemLoader(on_error="skip")

assay_ids = ['2631','624249','651741','588350','463230','492952','743219','492992','463213']
assay_id = assay_ids[1]
#assay_id = '624249' #bundled-safe assay example
size = 300
print(f"size: {size}")
use_equalized = True


limit_active = int(size // 2) if use_equalized else int(size)
limit_inactive = int(size // 2) if use_equalized else int(size)
graphs, targets = loader.load(
    assay_id,
    limit_active=limit_active,
    limit_inactive=limit_inactive,
)
targets = np.array(targets)

from abstractgraph.utils import plot_graph_label_counts
_ = plot_graph_label_counts(graphs, top=20, title='Dataset info', log_scale=True)


In [ ]:
from abstractgraph_graphicalizer.chem import draw_molecules as display_function

from abstractgraph.vectorize import AbstractGraphTransformer
from abstractgraph.operators import *
from abstractgraph_ml.feasibility import FeasibilityEstimatorFeatureCannotExist, FeasibilityEstimator

df = compose(neighborhood(radius=2), unlabel())
fe1 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)
df = add(neighborhood(radius=1), cycle())
fe2 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)
feasibility_estimators = [fe1, fe2]
feasibility_estimator = FeasibilityEstimator(feasibility_estimators)

nbits=14
#decomposition_function = compose(intersection_edges(), restore_label(), add(cycle(), tree()), unlabel())
#decomposition_function = add(cycle(), neighborhood(radius=(0,2)))
#decomposition_function = add(node(), edge(), cycle())
#decomposition_function = add(node(), compose_product(binary_combination(distance=3), neighborhood(), node()))
#decomposition_function = add(node(), edge(), cycle(), neighborhood(radius=(1,2)))
decomposition_function = neighborhood(radius=(0,2))

graph_transformer = AbstractGraphTransformer(
    nbits=nbits,
    decomposition_function=add(cycle(), neighborhood(radius=(0,2))),
    return_dense=True,
)
from abstractgraph_generative.autoregressive import AutoregressiveGraphGenerator
generator = AutoregressiveGraphGenerator(
    feasibility_estimator,  # Feasibility pipeline; filters invalid molecules per step.
    nbits,  # Bit width for hashing features in decomposition/vectorizers.
    decomposition_function,  # Operator defining interpretation-node associations for pruning/indexing.
    cut_radius=1,  # Compatibility radius for cut signatures (0=endpoints only; small → permissive).
    min_cut_size=2,  # Minimum virtual cut size; set to 2+ to discourage single attachments.
    max_cut_size=5,  # Maximum virtual cut size sampled during proposals.
    size_weights={1:0.1, 2: 1.0, 3: 1.0, 4: 0.5, 5: 0.5},  # Per-size sampling weights; downweight 1 to reduce filaments.
    prob_force_multicut=0.3,  # With this probability, restrict a proposal to multicut_sizes.
    multicut_sizes=(2,3,4,5),  # Sizes considered multi-attachment (encourage branching).
    max_virtual_cut_evaluations=10000,  # Per step: cap on sampled cut-node sets before matching/evaluating.
    cut_context_radius=None,  # Context radius for embeddings; None = use full outer/inner subgraphs.
    context_vectorizer=graph_transformer,  # Transformer to embed cut contexts for similarity.
    context_temperature=0.2,  # 0 = deterministic context ranking; >0 blends in uniform randomness.
    context_top_k=5,  # With temperature=0, sample uniformly from the top-k context-scored matches.
    n_virtual_candidates=1000,  # Number of rewrite proposals generated per growth step.
    use_context_embedding=True,  # Enable context-aware matching between source and donor cuts.
    min_nodes_for_pruning=2,  # Smallest fragment retained during pruning (controls donor pool granularity).
    similarity_vectorizer=graph_transformer,  # Vectorizer for whole-graph similarity in selection.
    similarity_k_neighbors=11,  # Average top-k cosine similarities to training graphs.
    use_similarity_selection=True,  # If False, fall back to edge-count heuristic instead of similarity.
    similarity_temperature=0.2,  # 0 = pick most similar; >0 softens selection with temperature.
    similarity_top_k=5,  # With temperature=0, sample from the top-k most similar candidates.
    n_pruning_iterations=300,  # Random pruning runs per training graph to build donor pool diversity.
    verbose=True,  # Print progress; when a display function is set, also show intermediate graphs.
    display_function=display_function,  # Callback to render molecules/graphs during generation.
    display_live=True,  # If True, stream intermediate displays; otherwise show them at the end.
    n_jobs=-1)  # Parallelism for donor construction and generation (-1 = all CPUs).

from abstractgraph.vectorize import AbstractGraphTransformer
from abstractgraph.operators import compose, neighborhood, cycle, add, unlabel
df = add(neighborhood(radius=(1,4)), cycle())
vec = AbstractGraphTransformer(nbits=nbits, decomposition_function=df, return_dense=True)
from abstractgraph_generative.repair import RepairGenerator
repair_generator = RepairGenerator(
    decomposition_function=df, 
    nbits=nbits, 
    graph_transformer=vec, 
    n_samples=128, 
    cut_radius=None, 
    cut_scope="outer", 
    replace_with_smaller_or_equal_size=True,
    single_replacement=True)

In [ ]:
from abstractgraph_graphicalizer.chem import draw_molecules as display_graphs
import random
import time

def _fmt_elapsed(seconds):
    """Format elapsed seconds as s, m, or h with two decimals.

    Args:
        seconds: Elapsed time in seconds.

    Returns:
        Human-readable duration string.
    """
    seconds = float(seconds)
    if seconds < 60:
        return f"{seconds:.2f}s"
    if seconds < 3600:
        return f"{seconds/60:.2f}m"
    return f"{seconds/3600:.2f}h"

def generate_graphs(graphs, generator, repair_generator, generator_size=1, n_samples=1, n_rounds=2, k_neighbors=7, n_iterations=1, verbose=True):
    generator.store(graphs)
    generator_graphs = generator.get_generators(size=generator_size)
    if verbose: 
        print('Generator graphs:')
        display_graphs(generator_graphs)
    t0 = time.perf_counter()
    generator.fit(generator_graphs)
    t1 = time.perf_counter()
    if verbose:
        print(f'Generator fit time: {_fmt_elapsed(t1 - t0)}')
    t0 = time.perf_counter()
    repair_generator.fit(graphs)
    t1 = time.perf_counter()
    if verbose:
        print(f'RepairGenerator fit time: {_fmt_elapsed(t1 - t0)}')
    samples = [generator.donors[random.randrange(len(generator.donors))] for it in range(n_samples)]
    if verbose: 
        print('Seed graphs:')
        display_graphs(samples)
    for _ in range(n_rounds):
        samples = generator.generate_from(samples)
        if verbose: 
            print('Generated graphs before repair:')
            display_graphs(samples)
        samples = repair_generator.repair(samples, k_neighbors=k_neighbors, n_iterations=n_iterations)
        if verbose: 
            print('Generated graphs after repair:')
            display_graphs(samples)
    samples = generator.generate_from(samples)
    if verbose: 
        print('Final graphs:')
        display_graphs(samples)
    if verbose: 
        print('Generator graphs:')
        display_graphs(generator_graphs)
    return samples

---

In [ ]:
%%time
samples = generate_graphs(graphs, generator, repair_generator, generator_size=3, n_samples=4, n_rounds=2, k_neighbors=7, n_iterations=1, verbose=True)

---